# Semantic Filtering & Reranking Pipeline Demo

This notebook demonstrates the new semantic filtering feature for the HybridSearch module.

## Overview

The semantic filtering feature allows you to filter SNOMED CT search results by **semantic categories** such as:
- `disorder` - Diseases and disorders
- `finding` - Clinical findings  
- `procedure` - Medical procedures
- `event` - Events
- `body structure` - Anatomical structures
- `substance` - Medicinal substances
- `organism` - Organisms

## Use Cases

1. **Clinical Decision Support**: Filter search results to show only disorders when looking for diagnosis terms
2. **EHR Integration**: Map clinical text to appropriate SNOMED semantic categories
3. **Specialty-Specific Searches**: A cardiology application might filter for 'finding' and 'procedure'


## Setup

In [1]:
import os
import sys

# Ensure project root is in Python path for local module imports
SCRIPT_DIR = (
    os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
)
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Configure paths
MEDCAT_PATH = (
    "/workspaces/snomed_methods/model_packs/medcat_model_pack_422d1d38fc58f158.zip"
)
UK_SNOMED_PATH = "/workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z"

print(f"UK SNOMED Path: {UK_SNOMED_PATH}")
print(f"MedCAT Model: {MEDCAT_PATH}")

UK SNOMED Path: /workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z
MedCAT Model: /workspaces/snomed_methods/model_packs/medcat_model_pack_422d1d38fc58f158.zip


In [2]:
# Import the required modules
from snomed_methods import HybridSearch, semantic_filter_results

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Initialize HybridSearch with SNOMED and MedCAT data
print("Initializing HybridSearch...")
searcher = HybridSearch(
    uk_path=UK_SNOMED_PATH,
    medcat_path=MEDCAT_PATH,
    backend="transformers",
    device="cpu",
)
print("✓ HybridSearch initialized")

Initializing HybridSearch...


/workspaces/snomed_methods/snomed_methods_env/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_md' (3.1.0) was trained with spaCy v3.1.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


✓ HybridSearch initialized


In [4]:
# Verify SEMANTIC_CATEGORIES configuration
print("Available semantic categories:")
for category, type_ids in searcher.SEMANTIC_CATEGORIES.items():
    print(f"  - {category}: {len(type_ids)} type IDs")

Available semantic categories:
  - disorder: 4 type IDs
  - finding: 2 type IDs
  - procedure: 3 type IDs
  - event: 3 type IDs
  - body structure: 3 type IDs
  - substance: 2 type IDs
  - organism: 2 type IDs
  - attribute: 2 type IDs
  - linkage concept: 2 type IDs
  - core concept: 1 type IDs


## Demo 1: Basic Search with Semantic Filter

In [5]:
# Perform a search for "diabetes" and filter to show only disorders
print("Searching for 'diabetes' with semantic filter: ['disorder']...")

results = searcher.search(
    query="diabetes",
    top_k=15,
    term_weight=0.3,
    hierarchy_weight=0.2,
    embedding_weight=0.5,
    semantic_filter=["disorder"],  # NEW: Filter by disorder category
)

print(f"\n✓ Found {len(results)} results")
print(f"  - Term matches: {results.term_matches}")
print(f"  - Hierarchy matches: {results.hierarchy_matches}")
print(f"  - Embedding matches: {results.embedding_matches}")

Searching for 'diabetes' with semantic filter: ['disorder']...

✓ Found 0 results
  - Term matches: 0
  - Hierarchy matches: 0
  - Embedding matches: 0


In [6]:
# Display the filtered results
import pandas as pd


def display_results(results, top_n=10) -> None:
    """Display search results in a formatted table."""
    data = []
    for cui, term, score in results.results[:top_n]:
        scores = results.cui_scores.get(cui, {})
        data.append(
            {
                "CUI": cui,
                "Term": term,
                "Combined Score": round(score, 4),
                "Term Score": scores.get("term", "-"),
                "Hierarchy Score": scores.get("hierarchy", "-"),
                "Embedding Score": scores.get("embedding", "-"),
            },
        )

    df = pd.DataFrame(data)
    print(df.to_string(index=False))


print("Top 10 Results (filtered to 'disorder' category):")
print("=" * 80)
display_results(results, top_n=10)

Top 10 Results (filtered to 'disorder' category):
Empty DataFrame
Columns: []
Index: []


## Demo 2: Multiple Semantic Categories

In [7]:
# Combine multiple semantic categories
print("Searching for 'hypertension' with filter: ['disorder', 'finding']...")

results_multi = searcher.search(
    query="hypertension",
    top_k=15,
    semantic_filter=["disorder", "finding"],  # Multiple categories
)

print(f"\n✓ Found {len(results_multi)} results")
display_results(results_multi, top_n=10)

Searching for 'hypertension' with filter: ['disorder', 'finding']...



✓ Found 0 results
Empty DataFrame
Columns: []
Index: []


## Demo 3: Using the Standalone Function

In [8]:
# Get results first, then filter them with the standalone function
print("Step 1: Get unfiltered results for 'stroke'")
results_unfiltered = searcher.search(
    query="stroke",
    top_k=20,
    term_weight=0.3,
    hierarchy_weight=0.2,
    embedding_weight=0.5,
)

print(f"  Unfiltered results: {len(results_unfiltered)}")

# Apply semantic filter separately
print("\nStep 2: Filter using standalone function for 'procedure' category")
results_filtered = semantic_filter_results(
    results=results_unfiltered,
    semantic_categories="procedure",  # Can be string or list
    uk_path=UK_SNOMED_PATH,
)

print(f"  Filtered results: {len(results_filtered)}")
display_results(results_filtered, top_n=10)

Step 1: Get unfiltered results for 'stroke'


  Unfiltered results: 20

Step 2: Filter using standalone function for 'procedure' category


  Filtered results: 0
Empty DataFrame
Columns: []
Index: []


## Demo 4: Practical Use Case - Clinical Specialty Search

In [9]:
# Example: Cardiology-focused search
print("Clinical Scenario: Cardiology specialist searching for heart-related terms")
print("=" * 80)

# Search for multiple cardiac terms with appropriate semantic filters
cardiac_terms = [
    "myocardial infarction",
    "arrhythmia",
    "congestive heart failure",
]

for term in cardiac_terms:
    print(f"\nSearching: '{term}'")
    results = searcher.search(
        query=term,
        top_k=5,
        semantic_filter=[
            "disorder",
            "finding",
        ],  # Cardiology is mostly disorders/findings
    )

    print(f"  Found {len(results)} results in 'disorder' and 'finding' categories")
    for i, (cui, term_name, score) in enumerate(results.results[:3], 1):
        print(f"    {i}. {term_name} (Score: {score:.3f})")

Clinical Scenario: Cardiology specialist searching for heart-related terms

Searching: 'myocardial infarction'


  Found 0 results in 'disorder' and 'finding' categories

Searching: 'arrhythmia'
  Found 0 results in 'disorder' and 'finding' categories

Searching: 'congestive heart failure'


  Found 0 results in 'disorder' and 'finding' categories


## Demo 5: Comparing Filtered vs Unfiltered Results

In [10]:
# Compare filtered vs unfiltered for "adenoma"
print("Comparing results for 'adenoma' (a tumor type)")
print("=" * 80)

# Unfiltered
results_unfiltered = searcher.search(
    query="adenoma",
    top_k=15,
    semantic_filter=None,  # No filter
)

# Filtered to show only disorders
results_filtered = searcher.search(
    query="adenoma",
    top_k=15,
    semantic_filter=["disorder"],
)

print(f"\nUnfiltered results: {len(results_unfiltered)}")
print("Top 5:")
for i, (cui, term, score) in enumerate(results_unfiltered.results[:5], 1):
    print(f"  {i}. {term}")

print(f"\nFiltered results: {len(results_filtered)}")
print("Top 5:")
for i, (cui, term, score) in enumerate(results_filtered.results[:5], 1):
    print(f"  {i}. {term}")

Comparing results for 'adenoma' (a tumor type)



Unfiltered results: 15
Top 5:
  1. [M]Borderline mucinous cystadenoma of the ovary (morphologic abnormality)
  2. [M]Borderline mucinous cystadenoma of the ovary 
  3. Congenital cystic adenomatoid malformation of the lung 
  4. [M]Respiratory tract adenomas and adenocarcinomas (morphologic abnormality)
  5. [M]Bronchial adenoma NOS (morphologic abnormality)

Filtered results: 0
Top 5:


## Demo 6: Filtering by Body Structure and Substance

In [11]:
# Example with body structures and substances
print("Searching for 'liver' with filter: ['body structure', 'substance']")

results = searcher.search(
    query="liver",
    top_k=15,
    semantic_filter=["body structure", "substance"],
)

print(f"\n✓ Found {len(results)} results")
display_results(results, top_n=10)

Searching for 'liver' with filter: ['body structure', 'substance']

✓ Found 0 results
Empty DataFrame
Columns: []
Index: []


## Summary & Key Takeaways

### What You Learned:

1. **Semantic Filtering**: Filter search results by SNOMED CT semantic categories
2. **Two API Options**:
   - `searcher.search(..., semantic_filter=[...])` - Pass during initial search
   - `semantic_filter_results(results, ...)` - Apply to existing results

3. **Available Categories**:
   - `disorder`, `finding`, `procedure`, `event`
   - `body structure`, `substance`, `organism`
   - Plus more (10 total)

### Practical Applications
- Clinical decision support systems
- EHR-based coding assistance
- Specialty-specific search interfaces
- Reducing noise in search results